In [27]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

DATA_PATH = "../data/raw/1429_1.csv"

df = pd.read_csv(DATA_PATH)
df.shape
df.head(5)

C:\Users\anita\AppData\Local\Temp\ipykernel_21316\196225757.py:8: DtypeWarning: Columns (0: name, 1: reviews.didPurchase) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


,id,name,asins,brand,categories,keys,manufacturer,reviews.date,reviews.dateAdded,reviews.dateSeen,...,reviews.doRecommend,reviews.id,reviews.numHelpful,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username
0,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,This product so far has not disappointed. My c...,Kindle,NaN,NaN,Adapter
1,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,great for beginner or experienced person. Boug...,very fast,NaN,NaN,truman
2,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,Inexpensive tablet for him to use and learn on...,Beginner tablet for our 9 year old son.,NaN,NaN,DaveZ
3,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-13T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,4.0,http://reviews.bestbuy.com/3545/5620406/review...,I've had my Fire HD 8 two weeks now and I love...,Good!!!,NaN,NaN,Shacks
4,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...","841667104676,amazon/53004484,amazon/b01ahb9cn2...",Amazon,2017-01-12T00:00:00.000Z,2017-07-03T23:33:15Z,"2017-06-07T09:04:00.000Z,2017-04-30T00:45:00.000Z",...,True,NaN,0.0,5.0,http://reviews.bestbuy.com/3545/5620406/review...,I bought this for my grand daughter when she c...,Fantastic Tablet for kids,NaN,NaN,explore42


In [36]:
df_processed = df.copy()

In [37]:
df_processed.isnull().sum()

id                          0
name                     6760
asins                       2
brand                       0
categories                  0
keys                        0
manufacturer                0
reviews.date               39
reviews.dateAdded       10621
reviews.dateSeen            0
reviews.didPurchase     34659
reviews.doRecommend       594
reviews.id              34659
reviews.numHelpful        529
reviews.rating             33
reviews.sourceURLs          0
reviews.text                1
reviews.title               6
reviews.userCity        34660
reviews.userProvince    34660
reviews.username            7
dtype: int64

In [38]:
#drop missing target
df_processed[["reviews.rating", "reviews.text"]].isna().sum()
df_processed["reviews.rating"].isna().sum()
df_processed = df_processed.dropna(subset=["reviews.rating"]) 

In [39]:
#Missing review text
df_processed["reviews.text"].isna().sum()
df_processed = df_processed.dropna(subset=["reviews.text"])

In [32]:
df_processed.shape

(34626, 21)

In [40]:
dup_key = ["asins", "reviews.username", "reviews.date", "reviews.text"]
print("duplicates on key columns:", df_processed.duplicated(subset=dup_key).sum())
df_processed = df_processed.drop_duplicates(subset=dup_key)
print(df_processed.shape)

duplicates on key columns: 0
(34626, 21)


In [41]:
# drop irrelvant columns
columns_to_drop = [
    "id",
    "reviews.userCity",
    "reviews.userProvince",
    "reviews.dateSeen",
    "reviews.dateAdded",
    "reviews.didPurchase",
    "reviews.id",
    "keys",
    "reviews.sourceURLs",
    "reviews.username"
	]
df_processed = df_processed.drop(columns=columns_to_drop)
print(df_processed.shape)
print(df_processed.columns)

(34626, 11)
Index(['name', 'asins', 'brand', 'categories', 'manufacturer', 'reviews.date',
       'reviews.doRecommend', 'reviews.numHelpful', 'reviews.rating',
       'reviews.text', 'reviews.title'],
      dtype='str')


In [42]:
name_by_asin = (df_processed.dropna(subset=["name"])
                .groupby("asins")["name"]
                .agg(lambda s: s.mode().iloc[0]))
df_processed["product_name"] = df_processed["name"].fillna(df_processed["asins"].map(name_by_asin))

print("names still missing:", df_processed["product_name"].isna().sum())




names still missing: 6088


In [43]:
#Handle missing product metadata
text_metadata_columns = ["product_name", "asins", "brand", "categories", "manufacturer", "reviews.title"]
   
for column in text_metadata_columns:
    df_processed[column] = df_processed[column].fillna("Unknown")

In [44]:
## rough
df_processed.isnull().sum()

name                   6759
asins                     0
brand                     0
categories                0
manufacturer              0
reviews.date             29
reviews.doRecommend     560
reviews.numHelpful      495
reviews.rating            0
reviews.text              0
reviews.title             0
product_name              0
dtype: int64

In [ ]:
# Handling missing data
df_processed["reviews.numHelpful"] = (
    df_processed["reviews.numHelpful"].fillna(0)
)

df_processed["reviews.doRecommend"] = (
    df_processed["reviews.doRecommend"].fillna(False)
)

df_processed.dtypes
df_processed["reviews.date"] = (
    df_processed["reviews.date"].fillna("Unknown")
)
print("Remaining missing values:",
      df_processed.isna().sum().sum())

Remaining missing values: 0


In [20]:
## rough
df_processed.shape
df_processed.duplicated().sum()

np.int64(0)

In [ ]:
def rating_to_sentiment(rating):
    if rating <= 2:
        return "negative"
    elif rating == 3:
        return "neutral"
    else:
        return "positive"

In [23]:
df_processed["sentiment"] = (
    df_processed["reviews.rating"]
    .apply(rating_to_sentiment)
)
df_processed[
    ["reviews.rating", "sentiment"]
].head()

,reviews.rating,sentiment
0,5.0,positive
1,5.0,positive
2,5.0,positive
3,4.0,positive
4,5.0,positive


In [24]:
#Validate sentiment mapping
pd.crosstab(
    df_processed["reviews.rating"],
    df_processed["sentiment"]
)

sentiment,negative,neutral,positive
reviews.rating,,,
1.0,410,0,0
2.0,402,0,0
3.0,0,1499,0
4.0,0,0,8541
5.0,0,0,23774


In [ ]:
#Check class distribution
df_processed["sentiment"].value_counts()
df_processed["sentiment"].value_counts(normalize=True) * 100

sentiment
positive    93.325825
neutral      4.329117
negative     2.345059
Name: proportion, dtype: float64